In [ ]:
# %pip install -q \
#     --extra-index-url=https://pypi.nvidia.com \
#     "cudf-cu12==25.6.*" "dask-cudf-cu12==25.6.*" "cuml-cu12==25.6.*" \
#     "cugraph-cu12==25.6.*" "nx-cugraph-cu12==25.6.*" "cuxfilter-cu12==25.6.*" \
#     "cucim-cu12==25.6.*" "pylibraft-cu12==25.6.*" "raft-dask-cu12==25.6.*" \
#     "cuvs-cu12==25.6.*" "nx-cugraph-cu12==25.6.*"

In [ ]:
# %load_ext cudf.pandas
import pandas as pd
import requests
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import re

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110 Safari/537.36"
}

In [ ]:
SITEMAPURL = "https://baodautu.vn/sitemap.xml"


def get_urls(outputpath: str):

    print("[INFO] Retrieving main sitemap")
    sitemap_response = requests.get(SITEMAPURL, timeout=10, headers=headers)
    sitemap_response.raise_for_status()

    print("[INFO] Retrieved main sitemap")
    root = ET.fromstring(sitemap_response.content)

    sitemaps = []
    pattern = re.compile(r"https://baodautu\.vn/sitemaps/news-(\d{4})-(\d{1,2})\.xml")

    for sitemap in root.findall(".//{*}sitemap"):
        loc = sitemap.find("{*}loc")
        if loc is not None:
            match = pattern.match(loc.text)
            if match:
                year = int(match.group(1))
                if year >= 2010 and year < 2025:
                    sitemaps.append(loc.text)
    print(f"[INFO] Fetched {len(sitemaps)} sitemap(s)")

    urls = []
    for sitemap in sitemaps:
        response = requests.get(sitemap.strip(), timeout=10, headers=headers)
        response.raise_for_status()
        
        root = ET.fromstring(response.content)
        for url in root.findall(".//{*}url"):
            loc = url.find("{*}loc")
            if loc is not None and loc.text and loc.text != "https://baodautu.vn":
                urls.append(loc.text.strip())
    print(f"[INFO] Fetched {len(urls)} article(s)")
                
    with open(outputpath, 'w') as f:
        for u in urls:
            f.write(u + "\n")
    print(f"Saved to {outputpath}")

In [ ]:
# get_urls("../data/raw/scraped/urls.txt")
with open("../data/raw/scraped/urls.txt", 'r') as f:
    urls = [line.strip() for line in f]

In [ ]:
def fetch_article(url: str):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    title = soup.find("div", class_="title-detail").get_text(strip=True)
    category = soup.find("div", class_="fs16 text-uppercase").get_text(strip=True)
    
    author_div = soup.find("div", class_="author-share-top")
    time = author_div.find("span", class_="post-time").get_text(strip=True)
    time = time.replace('-', '')
    time = time.strip()
    
    content_div = soup.find("div", id="content_detail_news")
    
    all_ps = content_div.find_all("p", recursive=False)
    content = "\n".join([p.get_text(strip=True) for p in all_ps])
    
    tags = soup.find_all("a", class_="tag_detail_item")
    tags = [tag.get_text(strip=True) for tag in tags]
    
    return {"url": url, "title": title, "category": category, "time": time, "content": content, "tags": tags}

In [ ]:
batch_number = 0
total_batches = 3
articles = []
failed_urls = []

batch_size = len(urls) // total_batches
start_index = batch_number * batch_size

# Ensure the last batch takes the remainder
if batch_number == total_batches - 1:
    end_index = len(urls)
else:
    end_index = (batch_number + 1) * batch_size

urls_to_scrape = urls[start_index:end_index]

for url in urls_to_scrape:
    try:
        article = fetch_article(url)
        articles.append(article)
        if len(articles) % 100 == 0:
            print(f"[INFO] Scraped {len(articles)}")
    except Exception as e:
        print(f"[ERROR] Failed to fetch {url}: {e}")
        failed_urls.append(url)
        continue

df = pd.DataFrame(articles)
if not df.empty and 'time' in df.columns:
    try:
        df['time'] = pd.to_datetime(df['time'], format='%d/%m/%Y %H:%M')
    except Exception as e:
        print(f"[WARNING] Failed to parse 'time' column: {e}")
    df.to_csv(f"../data/raw/scraped/articles_batch_{batch_number}.csv", index=False)

with open(f"../data/raw/scraped/failed_urls_batch_{batch_number}.txt", "w") as f:
    for u in failed_urls:
        f.write(u + "\n")